# Track A — SSD MobileNet v2 FPNLite 320×320 → TFLite INT8

**Proyecto:** `embebidos-3` — clasificador de residuos en Jetson Nano B01
**Plataforma:** Google Colab T4 con Python 3.10 (vía `condacolab`) + TF 2.15.0. Kaggle NO está soportado en este track porque TF Object Detection API choca con `/kaggle/input` read-only.
**Pre-requisito:** Roboflow Version 1-A generada con `Fit (black edges) in 320×320`, export `tfrecord`.
**Salida:** `detect_int8.tflite` (~3 MB) listo para inferencia en Jetson Nano (CPU + XNNPACK + NEON SIMD).

---

## Stack training vs runtime (mayo 2026)

| | Training (Colab) | Runtime (Jetson Nano JetPack 4.6.1) |
|---|---|---|
| Python | 3.10 (vía `condacolab`) | 3.6.9 |
| TF | 2.15.0 | 2.5.0+nv21.8 |
| OD API | research/object_detection (legacy, último TF compatible 2.15) | — |
| Cuantización | PTQ post-train con `representative_dataset` | TFLite Interpreter + XNNPACK + NEON |
| Hardware | T4 GPU | Cortex-A57 4 cores + Maxwell 128 CUDA cores |

Decisión completa: ver `investigaciones/2026-05-12/2026-05-12-compatibilidad-notebooks-training.md` (sección "Decisión QAT Track A").

## Por qué PTQ (no QAT) en este track

Originalmente este notebook activaba QAT vía `graph_rewriter.quantization` en `pipeline.config`. **Eso no funciona en TF 2.x.** Issue [tensorflow/models #9835](https://github.com/tensorflow/models/issues/9835) (open desde 2021-03-25) confirma verbatim: *"The graph_rewriter is not handled in tf2."* Las alternativas reales de QAT en TF 2.x son: (a) MediaPipe Model Maker `MOBILENET_V2_I320` con QAT preintegrado en checkpoint Model Garden, pero **omite `TFLite_Detection_PostProcess`** en export y requiere decoder custom en Nano; (b) TFMOT Keras-style, pero **no soporta SSD con FPN** out-of-the-box. Por eso elegimos PTQ con representative dataset (400 muestras del val) sobre el modelo entrenado FP32.

**Trade-off cuantificado:** Jacob et al. CVPR 2018 ([arXiv:1712.05877](https://arxiv.org/abs/1712.05877)) establece que QAT mantiene caída < 1.5 pp mAP; Karimov et al. 2025 ([arXiv:2508.19600](https://arxiv.org/abs/2508.19600)) mide PTQ INT8 con caída 3-7 pp mAP50-95. Aceptable en dataset waste-3class por: (1) 3 clases visualmente distintas (vidrio brillante / papel mate / plástico variado); (2) representative dataset bien calibrado de 400 muestras seed-fijada.

## Por qué CPU en Nano y no GPU Maxwell

La GPU Maxwell del Nano B01 (128 CUDA cores) **no tiene tensor cores INT8** — estos llegaron con Turing/Ampere. En Maxwell el INT8 acelera vía instrucciones SIMD packed del ARM Cortex-A57, no por GPU. Por eso el target runtime es **TFLite + XNNPACK + NEON**, NO TensorRT. Para FP16 con GPU está el Track B (YOLOv8).

## FPS esperado en Nano CPU

Estimación a partir de `NobuoTsukamoto/benchmarks` (Raspberry Pi 4 Cortex-A72 INT8 4 hilos): SSD MobileNet v2 plain 320 INT8 ≈ **14-18 FPS** en Cortex-A57 del Nano. FPNLite ≈ 8-10 FPS (margen ajustado). Si el threshold de 10 FPS no se cumple con FPNLite, pivot a SSD plain (mismo pipeline, distinto checkpoint base).


## 0. Detección de entorno y validación de pre-requisitos

Auto-detecta Colab/Kaggle/local, valida GPU/RAM/disco y muestra banner.

In [ ]:
import os, sys, platform, shutil, subprocess, time, json
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)
PLATFORM = "Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local"

def banner(title: str) -> None:
    print("\n" + "=" * 64); print(f"  {title}"); print("=" * 64)

def t_ts() -> str:
    return time.strftime("%H:%M:%S")

banner("Track A · SSD MobileNet v2 FPNLite 320x320 -> TFLite INT8")
print(f"[{t_ts()}] Plataforma   : {PLATFORM}")
print(f"[{t_ts()}] Python       : {sys.version.split()[0]} ({platform.machine()})")
print(f"[{t_ts()}] OS           : {platform.platform()}")

gpu_info = ""
try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        capture_output=True, text=True, timeout=5,
    )
    gpu_info = out.stdout.strip()
    print(f"[{t_ts()}] GPU          : {gpu_info or 'no detectada'}")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print(f"[{t_ts()}] GPU          : nvidia-smi no disponible")

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    print(f"[{t_ts()}] RAM total    : {ram_gb:.1f} GB")
except ImportError:
    ram_gb = 0.0

root_for_disk = "/content" if IS_COLAB else "/kaggle" if IS_KAGGLE else "."
disk_free_gb = shutil.disk_usage(root_for_disk).free / (1024 ** 3)
print(f"[{t_ts()}] Disco libre  : {disk_free_gb:.1f} GB en {root_for_disk}")

issues = []
if not gpu_info:
    issues.append("Sin GPU. Runtime -> Change runtime -> GPU (T4).")
if disk_free_gb < 15:
    issues.append(f"Solo {disk_free_gb:.1f} GB libres; se requieren ~15 GB.")
if IS_KAGGLE:
    issues.append("Este notebook NO está validado en Kaggle (TF OD API choca con /kaggle/input).")

if issues:
    print("\nWARNINGS:")
    for w in issues: print(f"  ! {w}")
else:
    print(f"\n[{t_ts()}] Pre-flight OK.")


## 1. Configuración y gestión de secrets

Cascada de obtención de `ROBOFLOW_API_KEY`: env var → Colab Secrets → Kaggle Secrets → getpass interactivo.

In [ ]:
from getpass import getpass

# === Parámetros del experimento ===
WORKSPACE          = "embebidos3"
PROJECT            = "waste-3class-lwld8"
VERSION            = 1            # Version 1-A: Fit-black 320x320, export tfrecord
NUM_TRAIN_STEPS    = 12000
BATCH_SIZE         = 16
REPR_DATASET_SIZE  = 400          # PTQ representative samples
EVAL_SEED          = 42
HEARTBEAT_SECS     = 30           # cadencia del log durante training

# === Helper de secrets ===
def get_secret(name: str, prompt: str | None = None) -> str:
    if name in os.environ and os.environ[name]:
        print(f"  [{t_ts()}] {name} <- env var"); return os.environ[name]
    if IS_COLAB:
        try:
            from google.colab import userdata
            v = userdata.get(name)
            if v:
                os.environ[name] = v
                print(f"  [{t_ts()}] {name} <- Colab Secrets"); return v
        except Exception as e:
            print(f"  [{t_ts()}] Colab Secrets fallo: {e}")
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            v = UserSecretsClient().get_secret(name)
            if v:
                os.environ[name] = v
                print(f"  [{t_ts()}] {name} <- Kaggle Secrets"); return v
        except Exception as e:
            print(f"  [{t_ts()}] Kaggle Secrets fallo: {e}")
    v = getpass(prompt or f"Pega {name}: ")
    os.environ[name] = v
    return v

banner("1. Configuración + secrets")
ROBOFLOW_API_KEY = get_secret("ROBOFLOW_API_KEY", "Pega tu Roboflow API Key: ")

print(f"\nResumen del experimento:")
for k, v in {
    "workspace": WORKSPACE, "project": PROJECT, "version": VERSION,
    "num_train_steps": NUM_TRAIN_STEPS, "batch_size": BATCH_SIZE,
    "quantization": "PTQ post-train (representative_dataset)",
    "repr_dataset_size": REPR_DATASET_SIZE,
    "eval_seed": EVAL_SEED, "heartbeat_secs": HEARTBEAT_SECS,
}.items():
    print(f"  {k:18s}= {v}")


## 2. Persistencia de artefactos

Montar Google Drive si estamos en Colab. Define `WORK_DIR` (filesystem efímero rápido) y `PERSIST_DIR` (Drive, supervive a desconexiones).

In [ ]:
banner("2. Persistencia")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PERSIST_ROOT = Path("/content/drive/MyDrive/embebidos-3/track_a")
    WORK_DIR     = Path("/content/track_a")
elif IS_KAGGLE:
    PERSIST_ROOT = Path("/kaggle/working/track_a")
    WORK_DIR     = Path("/kaggle/working/track_a_scratch")
else:
    PERSIST_ROOT = Path.cwd() / "out_track_a"
    WORK_DIR     = Path.cwd() / "scratch_track_a"

PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Subcarpetas
DATASET_DIR = WORK_DIR / "ds_tfr"
PRETRAINED_DIR = WORK_DIR / "pretrained"
RUNS_DIR = WORK_DIR / "runs"
EXPORT_DIR = WORK_DIR / "tflite_export"
for d in (DATASET_DIR, PRETRAINED_DIR, RUNS_DIR, EXPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Si en Drive ya hay checkpoints previos, los copiamos al scratch para reanudar
drive_runs = PERSIST_ROOT / "runs"
if drive_runs.exists() and any(drive_runs.iterdir()):
    print(f"  [{t_ts()}] Reanudación: copio checkpoints previos {drive_runs} -> {RUNS_DIR}")
    shutil.copytree(drive_runs, RUNS_DIR, dirs_exist_ok=True)

print(f"\n  WORK_DIR     : {WORK_DIR}    (efímero, rápido)")
print(f"  PERSIST_ROOT : {PERSIST_ROOT} (Drive / Kaggle output)")
print(f"  DATASET_DIR  : {DATASET_DIR}")
print(f"  RUNS_DIR     : {RUNS_DIR}")


## 3. Instalación de dependencias

Stack training pinneado: Python 3.10 (vía `condacolab` para bajar desde Colab Py 3.12), TF 2.15.0 (último compatible con TF OD API legacy), tf-models-official 2.15.0, protobuf 3.20.3, Pillow 10.4 (Pillow 12 rompe OD API). Si `condacolab.install()` se ejecuta, el kernel se reinicia automáticamente y hay que re-ejecutar esta celda.

In [ ]:
# Bootstrap defensivo: tras kernel restart, las funciones/vars de celdas previas
# (banner, t_ts, IS_COLAB) no estan en memoria. Definir fallbacks idempotentes
# para permitir re-ejecutar SOLO esta celda en el flujo de 2 restarts.
import os, sys, subprocess, time, shutil

if "banner" not in dir():
    def banner(s):
        line = "=" * 64
        print("")
        print(line)
        print(f"  {s}")
        print(line)

if "t_ts" not in dir():
    def t_ts():
        return time.strftime("%H:%M:%S")

if "IS_COLAB" not in dir():
    IS_COLAB = "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG") is not None

banner("3. Dependencias")
t0 = time.time()

# === Stack de training (Colab) vs runtime target (Jetson Nano JetPack 4.6.1) ===
#
# RUNTIME TARGET (inmutable, Jetson Nano B01):
#   Python 3.6.9, TF 2.5.0+nv21.8 wheel NVIDIA, TFLite_Detection_PostProcess incluido,
#   CUDA 10.2, cuDNN 8.2.1, TRT 8.2.1, OpenCV 4.1.1.
#   GPU Maxwell 128 CUDA cores SIN tensor cores INT8.
#
# STACK TRAINING (decision 2026-05-12 Ronda 2 v3):
#   Python 3.10 via condacolab + mamba downgrade.
#   FLUJO DE 2 RESTARTS (validado empiricamente 2026-05-12 en Colab):
#     Run 1 (pre-condacolab): instala condacolab + Miniforge -> reinicia kernel.
#     Run 2 (post-restart 1): detecta Python != 3.10 -> mamba install python=3.10 -> reinicia.
#     Run 3 (post-restart 2): Python 3.10 OK -> instala deps TF 2.15.
#   Razon: condacolab.install_from_url() con Miniforge 23.11.0 NO baja Python a 3.10
#   (observado empiricamente; el installer puede traer Python 3.12 o el wrapper no
#   exec correctamente). Solucion robusta: forzar downgrade post-restart con mamba.
#   condacolab.check() lanza AssertionError si conda no esta (no devuelve False);
#   install_from_url(run_checks=True) maneja AssertionError internamente.
#
#   TF 2.15.0 (ultimo compatible con TF Object Detection API legacy; TF 2.16+
#   rompe tf.estimator — issue tensorflow/models #13599).
#   tf-models-official 2.15.0, Pillow 10.4, protobuf 3.20.3.
#
# QAT pivot a PTQ:
#   graph_rewriter en pipeline.config NO funciona en TF2 (issue tensorflow/models #9835).
#   PTQ con representative_dataset en conversion TFLite (celda 12). Caida 3-7 pp mAP50-95
#   segun Karimov et al. 2025 (arXiv:2508.19600).

def run_pip(args: list[str]) -> None:
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print(f"  [{t_ts()}] pip install {' '.join(args)}")
    subprocess.run(cmd, check=True)

MINIFORGE_URL = (
    "https://github.com/conda-forge/miniforge/releases/download/"
    "23.11.0-0/Miniforge3-23.11.0-0-Linux-x86_64.sh"
)

conda_in_path = shutil.which("conda") is not None
python_is_310 = sys.version_info[:2] == (3, 10)

if IS_COLAB and not python_is_310:
    print(f"  [{t_ts()}] Python actual: {sys.version.split()[0]} (conda_in_path={conda_in_path})")

    if not conda_in_path:
        # === RUN 1 (pre-condacolab) ===
        # Conda no esta. Instalar condacolab y Miniforge. Reinicia kernel.
        print(f"  [{t_ts()}] Etapa 1/2: instalando condacolab + Miniforge 23.11.0...")
        try:
            import condacolab
        except ImportError:
            run_pip(["condacolab"])
            import condacolab
        condacolab.install_from_url(MINIFORGE_URL)
        # install_from_url llama get_ipython().kernel.do_shutdown(True) al final.
        # do_shutdown es asincrono — el codigo aqui puede ejecutarse antes del kill.
        # Forzar fin de celda con sys.exit para evitar ejecutar el codigo de deps
        # con el Python viejo (3.12) antes de que muera el kernel.
        print(f"  [{t_ts()}] Kernel reiniciando — RE-EJECUTA esta celda tras restart.")
        sys.exit(0)
    else:
        # === RUN 2 (post-restart 1, conda instalado pero Python != 3.10) ===
        # Forzar downgrade con mamba. Mamba resuelve deps que necesitan rebuild
        # para Python 3.10 (matplotlib-base, psutil, google-colab).
        print(f"  [{t_ts()}] Etapa 2/2: Python {sys.version.split()[0]} != 3.10. Forzando mamba install python=3.10...")
        print(f"  [{t_ts()}] (puede tardar 3-7 min porque mamba resuelve deps)")
        subprocess.run(
            ["mamba", "install", "-y", "-c", "conda-forge", "python=3.10",
             "matplotlib-base", "psutil", "google-colab"],
            check=True,
        )
        print(f"  [{t_ts()}] Downgrade completo. Reiniciando kernel manualmente...")
        # Reiniciar kernel manualmente para que el nuevo Python tome efecto.
        try:
            import IPython
            IPython.get_ipython().kernel.do_shutdown(True)
        except Exception as _e:
            print(f"  [{t_ts()}] do_shutdown fallo ({_e}). Reinicia el kernel manualmente: Runtime > Restart runtime.")
        print(f"  [{t_ts()}] RE-EJECUTA esta celda tras el segundo restart.")
        sys.exit(0)

# === RUN 3 (post-restart 2 o local con Python 3.10) ===
# Python 3.10 confirmado. Proceder con deps.
if not python_is_310 and IS_COLAB:
    raise RuntimeError(
        f"Stack inesperado: Python {sys.version.split()[0]} en Colab tras 2 restarts. "
        "Workaround manual: ejecutar en celda nueva "
        "`!mamba install -y -c conda-forge python=3.10` + Runtime > Restart runtime. "
        "Plan B documentado en investigaciones/2026-05-12/2026-05-12-compatibilidad-notebooks-training.md."
    )

print(f"  [{t_ts()}] Python {sys.version.split()[0]} OK. Procediendo con deps.")

# Red de seguridad: forzar implementacion Python pura de protobuf.
# Mitiga error "cannot import name 'runtime_version' from 'google.protobuf'".
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# Deps del stack TF OD API legacy. Pinneados conservadores.
deps = [
    "tensorflow==2.15.0",
    "tensorflow-text==2.15.0",
    "tf-models-official==2.15.0",
    "tensorflow-model-optimization>=0.7.5,<0.8.0",
    "protobuf==3.20.3",
    "numpy==1.26.4",
    "Pillow==10.4.0",
    "opencv-python-headless==4.10.0.84",
    "pycocotools==2.0.7",
    "lvis==0.5.3",
    "tqdm",
    "roboflow>=1.3.6,<1.4",
]
install_failures = []
for pkg in deps:
    try:
        run_pip([pkg])
    except subprocess.CalledProcessError as e:
        print(f"  ! Fallo instalando {pkg}: {e}")
        install_failures.append(pkg)

if IS_COLAB:
    subprocess.run(["apt-get", "install", "-y", "-q", "protobuf-compiler"], check=False)

print(f"\n  [{t_ts()}] Dependencias listas en {time.time()-t0:.1f}s")

critical = {"tensorflow==2.15.0", "tf-models-official==2.15.0", "protobuf==3.20.3"}
critical_failed = [p for p in install_failures if p in critical]
if critical_failed:
    raise RuntimeError(
        f"Deps criticas fallaron: {critical_failed}. "
        f"Python actual: {sys.version.split()[0]} (TF 2.15 requiere 3.7-3.11)."
    )

# Re-pin defensivo Pillow + protobuf.
print(f"  [{t_ts()}] Re-pin defensivo Pillow 10.4 + protobuf 3.20.3 post-deps...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
     "Pillow==10.4.0", "protobuf==3.20.3"],
    check=True,
)

import importlib
expected = {
    "tensorflow": "2.15.",
    "tensorflow_text": "2.15.",
    "tensorflow_model_optimization": "0.7.",
    "google.protobuf": "3.20.",
    "PIL": "10.4.",
    "cv2": "4.10.",
    "roboflow": "1.3.",
}
print(f"\n  Versiones instaladas:")
for mod, prefix in expected.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        ok = "OK" if str(ver).startswith(prefix) else "!!"
        print(f"    {ok} {mod:32s}{ver}  (esperado {prefix}*)")
    except ImportError as e:
        print(f"    XX {mod}: {e}")


## 4. Setup TensorFlow Object Detection API

Clonar `tensorflow/models`, compilar protos, instalar el paquete. Skip si ya está hecho (resume seguro).

In [ ]:
banner("4. TF Object Detection API")

# Pin a commit SHA de master (NO usar tag v2.15.0: ese tag elimina research/).
# 9cafa3d150 = ultimo commit a research/object_detection del 2026-03-17, anterior
# al patch "Support Pillow 12" del 2026-04-29 (commit 971ded9e16) que rompe el
# pin Pillow 10.4. tag v2.15.0 (oct 2023) solo contiene official/ + orbit/.
# Verificado: gh api repos/tensorflow/models/commits?sha=master&path=research/object_detection
TF_MODELS_DIR = WORK_DIR / "models"
TF_MODELS_SHA = "9cafa3d150"

if not (TF_MODELS_DIR / ".git").exists():
    print(f"  [{t_ts()}] git clone tensorflow/models (master, sera checkout a {TF_MODELS_SHA})...")
    # No usamos --depth 1 + --branch SHA porque git no soporta shallow clone con SHA.
    # Clone superficial + fetch del SHA especifico es lo mas economico.
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout",
         "https://github.com/tensorflow/models.git", str(TF_MODELS_DIR)],
        check=True,
    )
    subprocess.run(["git", "-C", str(TF_MODELS_DIR), "checkout", TF_MODELS_SHA], check=True)
else:
    print(f"  [{t_ts()}] tensorflow/models ya clonado en {TF_MODELS_DIR}")

# Confirmar que research/ existe (proteccion contra checkout fallido)
research_dir = TF_MODELS_DIR / "research"
assert (research_dir / "object_detection").exists(), (
    f"research/object_detection no existe en {research_dir}. "
    f"Verifica que el SHA {TF_MODELS_SHA} este vigente."
)

# Compilar protos con grpcio-tools <= 1.64.1 EXPRESO. Versiones >= 1.66 generan
# *_pb2.py con `from google.protobuf import runtime_version` que solo existe en
# protobuf >= 4.x, y nuestro pin es protobuf 3.20.3 (issue tensorflow/models #11192,
# confirmado por @namasSinjali y SO #78671850).
print(f"  [{t_ts()}] pip install grpcio-tools==1.64.1 (compatible con protobuf 3.20.x)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "grpcio-tools==1.64.1"],
    check=True,
)

proto_files = list((research_dir / "object_detection" / "protos").glob("*.proto"))
print(f"  [{t_ts()}] Compilando {len(proto_files)} archivos .proto con grpc_tools.protoc...")
subprocess.run(
    [sys.executable, "-m", "grpc_tools.protoc",
     "-I" + str(research_dir),
     "--python_out=" + str(research_dir),
     *[str(p) for p in proto_files]],
    check=True,
)

setup_src = research_dir / "object_detection" / "packages" / "tf2" / "setup.py"
setup_dst = research_dir / "setup.py"
if setup_src.exists() and not setup_dst.exists():
    shutil.copy(setup_src, setup_dst)

print(f"  [{t_ts()}] pip install --no-deps -e ./models/research (puede tardar 1-2 min)...")
# --no-deps evita que el setup.py de OD API arrastre apache-beam/google-api-core
# que tiran protobuf >= 4.x (cadena documentada en research-web Ronda 2).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(research_dir)],
    check=True,
)

# Re-pin defensivo Pillow + protobuf DESPUES del install (red de seguridad).
print(f"  [{t_ts()}] Re-pin defensivo Pillow 10.4 + protobuf 3.20.3...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
     "Pillow==10.4.0", "protobuf==3.20.3"],
    check=True,
)

# Red de seguridad #2: forzar parser Python puro de protobuf. Si algun *_pb2.py
# residual (de un wheel cacheado) tira de `runtime_version`, esta env var evita
# el crash usando la implementacion Python en vez del C++ binding.
# Costo: protobuf parser ~5-10x mas lento, pero training overhead es negligible
# (los protos se leen una vez al levantar pipeline.config).
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# Red de seguridad #3: shim de Pillow para codigo que aun usa Image.ANTIALIAS
# (removido en Pillow 10.0). research/object_detection/utils/visualization_utils.py
# tiene fallback pero algunos paths legacy lo invocan.
try:
    from PIL import Image as _PILImage
    if not hasattr(_PILImage, "ANTIALIAS"):
        _PILImage.ANTIALIAS = _PILImage.LANCZOS
        print(f"  [{t_ts()}] Shim PIL.Image.ANTIALIAS -> LANCZOS aplicado")
except ImportError:
    pass

# Test del import (este import tira de protobuf, valida el stack entero)
try:
    from object_detection import model_lib_v2  # noqa: F401
    print(f"  [{t_ts()}] object_detection import OK")
except Exception as e:
    print(f"  [{t_ts()}] object_detection import fallo: {e}")
    raise


## 5. Descarga del dataset Roboflow

Skip si ya existe. Validación de estructura (label_map, splits, conteo bbox por clase).

In [ ]:
banner("5. Dataset Roboflow")

import glob
import re as _re
import zipfile

def _label_map_classes(path: Path) -> set[str] | None:
    """Parse _label_map.pbtxt y devuelve set de clases lowercase, o None si no parsea."""
    try:
        txt = path.read_text(errors="ignore")
        names = _re.findall(r"name:\s*['\"]([^'\"]+)['\"]", txt)
        if not names:
            return None
        return {n.strip().lower() for n in names}
    except OSError:
        return None

def find_all_label_maps() -> list[tuple[Path, set[str] | None]]:
    """Busca TODOS los _label_map.pbtxt en ubicaciones probables."""
    roots: set[Path] = {Path.cwd(), WORK_DIR, WORK_DIR.parent, Path.home()}
    extra = ["/kaggle/working", "/kaggle/working/datasets",
              "/content", "/content/datasets"]
    for p in extra:
        roots.add(Path(p))

    found: list[tuple[Path, set[str] | None]] = []
    seen: set[Path] = set()
    for root in roots:
        if not root.exists():
            continue
        try:
            for cand in root.glob("**/_label_map.pbtxt"):
                rp = cand.resolve()
                if rp in seen:
                    continue
                seen.add(rp)
                found.append((cand, _label_map_classes(cand)))
        except (PermissionError, OSError):
            continue
    return found

def pick_correct_label_map(candidates: list[tuple[Path, set[str] | None]]) -> Path | None:
    """Elige el _label_map.pbtxt con clases exactas {glass, paper, plastic}; fallback."""
    expected = {"glass", "paper", "plastic"}
    for path, classes in candidates:
        if classes == expected:
            return path
    for path, classes in candidates:
        if classes and len(classes & expected) >= 2:
            return path
    return candidates[0][0] if candidates else None

def extract_any_zips_in(root: Path) -> int:
    """Si el SDK dejó zips sin extraer, extraerlos. Devuelve cantidad extraída."""
    if not root.exists():
        return 0
    n = 0
    for z in list(root.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z) as zf:
                target = z.parent / z.stem
                target.mkdir(parents=True, exist_ok=True)
                zf.extractall(target)
            print(f"  [{t_ts()}] Zip extraído: {z} → {target}")
            n += 1
        except (zipfile.BadZipFile, OSError) as e:
            print(f"  ! No pude extraer {z}: {e}")
    return n

# CRITICAL: remover DATASET_DIRECTORY si está set — entra en conflicto con `location` arg
# (observado en Kaggle 2026-05-12 Track B: SDK reporta ds.location=X pero deja vacío).
if "DATASET_DIRECTORY" in os.environ:
    print(f"  [{t_ts()}] Removiendo DATASET_DIRECTORY env var (conflict con location arg)")
    del os.environ["DATASET_DIRECTORY"]

existing = next(iter(DATASET_DIR.glob("**/_label_map.pbtxt")), None)
ds = None
if existing is None:
    from roboflow import Roboflow

    print(f"  [{t_ts()}] DATASET_DIR : {DATASET_DIR}")
    print(f"  [{t_ts()}] CWD antes   : {Path.cwd()}")

    # Mitigación bug roboflow-python: chdir antes de download
    original_cwd = Path.cwd()
    os.chdir(DATASET_DIR)
    try:
        print(f"  [{t_ts()}] Descargando {WORKSPACE}/{PROJECT}/v{VERSION} (tfrecord)...")
        rf = Roboflow(api_key=ROBOFLOW_API_KEY)
        proj = rf.workspace(WORKSPACE).project(PROJECT)
        ds = proj.version(VERSION).download(
            "tfrecord", location=str(DATASET_DIR), overwrite=False
        )
        print(f"  [{t_ts()}] ds.location: {ds.location}")
        print(f"  [{t_ts()}] CWD después: {Path.cwd()}")
    finally:
        os.chdir(original_cwd)

    # Extraer zips no extraídos (defensivo)
    for diag_root in (DATASET_DIR, WORK_DIR, Path.cwd(),
                       Path("/kaggle/working") if Path("/kaggle/working").exists() else None,
                       Path("/content") if Path("/content").exists() else None):
        if diag_root:
            extract_any_zips_in(diag_root)

    # Búsqueda exhaustiva con regex parse del label_map
    print(f"\n  [{t_ts()}] Búsqueda exhaustiva de _label_map.pbtxt ...")
    all_maps = find_all_label_maps()
    print(f"  [{t_ts()}] Encontrados {len(all_maps)} archivos:")
    for path, classes in all_maps:
        cls_str = sorted(classes) if classes else "?"
        try:
            size = path.stat().st_size
        except OSError:
            size = -1
        print(f"    {path}  ({size} bytes, classes={cls_str})")

    label_map_file = pick_correct_label_map(all_maps)

    # Migrar contenido si no está en DATASET_DIR
    if label_map_file is not None and DATASET_DIR not in label_map_file.parents:
        src_dir = label_map_file.parent
        print(f"\n  [{t_ts()}] Migrando contenido desde {src_dir} → {DATASET_DIR}")
        for item in src_dir.iterdir():
            dst = DATASET_DIR / item.name
            if dst.exists():
                continue
            try:
                if item.is_dir():
                    shutil.copytree(item, dst)
                else:
                    shutil.copy2(item, dst)
            except (OSError, shutil.Error) as e:
                print(f"    ! No pude copiar {item.name}: {e}")
        label_map_file = next(iter(DATASET_DIR.glob("**/_label_map.pbtxt")), None)
else:
    label_map_file = existing
    print(f"  [{t_ts()}] Dataset ya presente en {DATASET_DIR}.")

if label_map_file is None:
    print(f"\n  ⚠️  _label_map.pbtxt NO encontrado.")
    diag_paths = [DATASET_DIR, WORK_DIR, Path.cwd(),
                  Path("/kaggle/working") if Path("/kaggle/working").exists() else None,
                  Path("/content") if Path("/content").exists() else None]
    for diag_root in [p for p in diag_paths if p and p.exists()]:
        print(f"\n  Contenido recursivo (primeros 30) de {diag_root}:")
        items = sorted(diag_root.rglob("*"))[:30]
        if not items:
            print(f"    (vacío)")
        else:
            for p in items:
                try:
                    rel = p.relative_to(diag_root)
                    kind = "d" if p.is_dir() else "f"
                    print(f"    [{kind}] {rel}")
                except Exception:
                    print(f"    {p}")
    raise FileNotFoundError(
        "Roboflow no produjo _label_map.pbtxt. Posible bug roboflow-python "
        "location ignored. Verifica API key y permisos del workspace."
    )

# DATASET_ROOT es el directorio real (padre de _label_map.pbtxt) — puede diferir de DATASET_DIR
DATASET_ROOT = label_map_file.parent
print(f"  [{t_ts()}] _label_map.pbtxt: {label_map_file}")
print(f"  [{t_ts()}] DATASET_ROOT    : {DATASET_ROOT}")

# Validación de estructura de splits — algunas Versions de Roboflow no incluyen test
splits = {}
for split in ("train", "valid", "test"):
    tfrs = sorted(glob.glob(str(DATASET_ROOT / split / "*.tfrecord")))
    splits[split] = tfrs
    print(f"  {split:6s}: {len(tfrs)} tfrecord(s)")

# train y valid son obligatorios; test es opcional (algunas versions sin test split)
assert splits["train"] and splits["valid"], \
    f"train/valid vacíos bajo {DATASET_ROOT}. Re-verifica Roboflow Version 1-A."
if not splits["test"]:
    print(f"  ⚠️  Sin test split — Track A funcionará pero smoke test fallback al valid.")

# Parse label_map: case-insensitive — Roboflow puede exportar capitalized
label_map_text = label_map_file.read_text()
classes_raw = _re.findall(r"name:\s*['\"]([^'\"]+)['\"]", label_map_text)
classes_lower = {c.strip().lower() for c in classes_raw}
print(f"\n  Clases detectadas ({len(classes_raw)}): raw={classes_raw}, lower={sorted(classes_lower)}")
expected = {"paper", "glass", "plastic"}
assert classes_lower == expected, \
    f"Esperaba (case-insensitive) {sorted(expected)}, encontré {sorted(classes_lower)}"
print(f"  [{t_ts()}] Dataset validado.")


## 6. Checkpoint pre-entrenado COCO

Skip si ya está. Descarga `ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8` (~12 MB).

In [ ]:
banner("6. Checkpoint pre-entrenado")

CKPT_NAME = "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8"
CKPT_URL  = f"http://download.tensorflow.org/models/object_detection/tf2/20200711/{CKPT_NAME}.tar.gz"
CKPT_DIR  = PRETRAINED_DIR / CKPT_NAME

if CKPT_DIR.exists() and (CKPT_DIR / "checkpoint" / "ckpt-0.index").exists():
    print(f"  [{t_ts()}] Checkpoint ya presente en {CKPT_DIR}")
else:
    tar_path = PRETRAINED_DIR / f"{CKPT_NAME}.tar.gz"
    print(f"  [{t_ts()}] Descargando {CKPT_URL}")
    import urllib.request
    from tqdm.auto import tqdm

    class _TqdmHook(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None: self.total = tsize
            self.update(b * bsize - self.n)

    with _TqdmHook(unit="B", unit_scale=True, miniters=1, desc="ckpt") as t_bar:
        urllib.request.urlretrieve(CKPT_URL, tar_path, reporthook=t_bar.update_to)

    print(f"  [{t_ts()}] Extrayendo...")
    subprocess.run(["tar", "xzf", str(tar_path), "-C", str(PRETRAINED_DIR)], check=True)
    tar_path.unlink()

assert (CKPT_DIR / "pipeline.config").exists(), "pipeline.config no encontrado."
print(f"  [{t_ts()}] Checkpoint listo: {CKPT_DIR}")


## 7. Construir `pipeline_custom.config`

Reescribe los campos críticos del pipeline original: 3 clases, `num_steps`, `batch_size`, paths a tfrecords/label_map/checkpoint.

**Sin `graph_rewriter`:** la cuantización se aplica como PTQ en la conversión a TFLite (celda 12). Activar `graph_rewriter.quantization` en TF 2.x es un no-op silencioso (issue tensorflow/models #9835).

In [ ]:
banner("7. Pipeline config (sin graph_rewriter, PTQ vía TFLite converter)")
import re as _re

config_template = (CKPT_DIR / "pipeline.config").read_text()
config = config_template

# 3 clases
config = _re.sub(r"num_classes:\s*\d+", "num_classes: 3", config)

# Eliminar graph_rewriter si existía (es placebo en TF2 — issue tensorflow/models #9835).
# La cuantización se aplica como PTQ en celda 12 vía TFLiteConverter +
# representative_dataset.
config = _re.sub(r"graph_rewriter\s*\{[^}]*\}", "", config, flags=_re.DOTALL)

# Paths
ckpt0 = str((CKPT_DIR / "checkpoint" / "ckpt-0").as_posix())
config = _re.sub(r'fine_tune_checkpoint:\s*".*"', f'fine_tune_checkpoint: "{ckpt0}"', config)
config = _re.sub(r'fine_tune_checkpoint_type:\s*".*"',
                 'fine_tune_checkpoint_type: "detection"', config)
config = _re.sub(r'label_map_path:\s*".*?"',
                 f'label_map_path: "{label_map_file.as_posix()}"', config)
config = _re.sub(
    r'(train_input_reader.*?input_path:\s*)".*?"',
    rf'\1"{(DATASET_ROOT / "train").as_posix()}/*.tfrecord"',
    config, flags=_re.DOTALL,
)
config = _re.sub(
    r'(eval_input_reader.*?input_path:\s*)".*?"',
    rf'\1"{(DATASET_ROOT / "valid").as_posix()}/*.tfrecord"',
    config, flags=_re.DOTALL,
)

# Hiperparámetros
config = _re.sub(r"num_steps:\s*\d+", f"num_steps: {NUM_TRAIN_STEPS}", config)
config = _re.sub(r"batch_size:\s*\d+", f"batch_size: {BATCH_SIZE}", config)

PIPELINE_PATH = WORK_DIR / "pipeline_custom.config"
PIPELINE_PATH.write_text(config)

print(f"  [{t_ts()}] {PIPELINE_PATH.name} listo ({PIPELINE_PATH.stat().st_size} bytes)")
print(f"\n  Verificación de campos clave:")
for needle in ("num_classes: 3", f"num_steps: {NUM_TRAIN_STEPS}",
               f"batch_size: {BATCH_SIZE}"):
    found = needle in config
    print(f"    {'✓' if found else '✗'} {needle}")
# Confirmación explícita: graph_rewriter ausente
gr_present = "graph_rewriter" in config
print(f"    {'✓' if not gr_present else '✗'} graph_rewriter ausente (PTQ via TFLite converter)")


## 8. TensorBoard inline

Arranca TB apuntando a `RUNS_DIR` para ver loss/learning_rate en vivo dentro del notebook.

In [ ]:
banner("8. TensorBoard inline")

TRAIN_RUN_DIR = RUNS_DIR / "track_a_ssd_v1"
TRAIN_RUN_DIR.mkdir(parents=True, exist_ok=True)

try:
    # noqa: ensure magic available
    get_ipython().run_line_magic("load_ext", "tensorboard")
    get_ipython().run_line_magic("tensorboard", f"--logdir {RUNS_DIR}")
    print(f"  [{t_ts()}] TensorBoard cargado. Recarga el dashboard cuando arranque training.")
except Exception as e:
    print(f"  ! TensorBoard inline no disponible: {e}")
    print(f"    Alternativa: !tensorboard --logdir {RUNS_DIR} --bind_all")


## 9. Entrenamiento con heartbeat

`model_main_tf2.py` no expone tqdm — lanzamos como subprocess y stremeamos su stderr, imprimiendo una línea de **heartbeat cada `HEARTBEAT_SECS` segundos** con:

- Step / num_steps + ETA
- Loss extraído del último `INFO`
- RAM y GPU memory en uso

Esto te garantiza ver "señal de vida" incluso si TF tarda 30-60s entre prints. Si la celda no muestra heartbeat por >120s, el training se colgó (Ctrl-C y reinvocar — los checkpoints se mantienen).

In [ ]:
banner("9. Training (con heartbeat)")

import threading
import re as _re2

TRAIN_CMD = [
    sys.executable,
    str(TF_MODELS_DIR / "research" / "object_detection" / "model_main_tf2.py"),
    f"--pipeline_config_path={PIPELINE_PATH}",
    f"--model_dir={TRAIN_RUN_DIR}",
    f"--num_train_steps={NUM_TRAIN_STEPS}",
    "--alsologtostderr",
]

state = {
    "last_step": 0, "last_loss": float("nan"),
    "started_at": time.time(), "lines": 0, "dead": False,
}

def _proc_memory_mb() -> float:
    try:
        import psutil
        return psutil.Process().memory_info().rss / (1024 ** 2)
    except Exception:
        return 0.0

def _gpu_mem_mb() -> tuple[int, int]:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=3,
        )
        used, total = map(int, out.stdout.strip().split(","))
        return used, total
    except Exception:
        return 0, 0

def heartbeat_loop() -> None:
    while not state["dead"]:
        time.sleep(HEARTBEAT_SECS)
        if state["dead"]: break
        elapsed = time.time() - state["started_at"]
        step = state["last_step"]
        loss = state["last_loss"]
        rss_mb = _proc_memory_mb()
        gpu_used, gpu_total = _gpu_mem_mb()
        eta_str = "n/a"
        if step > 0:
            speed = step / elapsed
            remaining = (NUM_TRAIN_STEPS - step) / speed if speed > 0 else 0
            eta_str = f"{remaining/60:.1f} min"
        print(f"  [HEARTBEAT {t_ts()}] step {step}/{NUM_TRAIN_STEPS}  "
              f"loss={loss:.3f}  RAM={rss_mb:.0f}MB  "
              f"GPU={gpu_used}/{gpu_total}MB  ETA~{eta_str}",
              flush=True)

print(f"  [{t_ts()}] Lanzando training...")
print(f"  CMD: {' '.join(TRAIN_CMD)}\n")

hb = threading.Thread(target=heartbeat_loop, daemon=True)
hb.start()

step_re = _re.compile(r"Step\s+(\d+)\s+per-step")
loss_re = _re.compile(r"Loss/total_loss[:\s=]+([\d.]+)")

proc = subprocess.Popen(
    TRAIN_CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

try:
    for line in proc.stdout:
        state["lines"] += 1
        ms = step_re.search(line)
        if ms: state["last_step"] = int(ms.group(1))
        ml = loss_re.search(line)
        if ml: state["last_loss"] = float(ml.group(1))
        # Echo selectivo: errores siempre; INFO cada 100 líneas
        upper = line.upper()
        if any(k in upper for k in ("ERROR", "TRACEBACK", "FAILED", "WARNING", "OOM")):
            print(line.rstrip())
        elif state["lines"] % 100 == 0:
            print(f"  [tail {state['lines']:>5}] {line.rstrip()}")
    rc = proc.wait()
finally:
    state["dead"] = True
    hb.join(timeout=2)

print(f"\n  [{t_ts()}] Training terminó con exit code {rc}")
assert rc == 0, "Training falló — revisar logs anteriores."
print(f"  [{t_ts()}] Total elapsed: {(time.time()-state['started_at'])/60:.1f} min")


## 10. Evaluación one-shot (sin colgar)

`model_main_tf2.py --checkpoint_dir` por defecto espera nuevos checkpoints en loop. Acá lo invocamos con `--eval_timeout=1` para que termine tras procesar el último.

In [ ]:
banner("10. Eval one-shot")

EVAL_CMD = [
    sys.executable,
    str(TF_MODELS_DIR / "research" / "object_detection" / "model_main_tf2.py"),
    f"--pipeline_config_path={PIPELINE_PATH}",
    f"--model_dir={TRAIN_RUN_DIR}",
    f"--checkpoint_dir={TRAIN_RUN_DIR}",
    "--eval_timeout=1",
    "--alsologtostderr",
]

t_eval = time.time()
try:
    eval_proc = subprocess.run(
        EVAL_CMD, capture_output=True, text=True, timeout=600
    )
    eval_out = (eval_proc.stdout or "") + "\n" + (eval_proc.stderr or "")
    map50 = None
    for line in eval_out.splitlines():
        if "DetectionBoxes_Precision/mAP@.50IOU" in line:
            m = _re.search(r"=\s*([\d.]+)", line)
            if m: map50 = float(m.group(1))
    if map50 is not None:
        print(f"  [{t_ts()}] mAP@.50 (FP32, val) = {map50:.4f}")
    else:
        print(f"  ! mAP no detectada en stdout. Últimas 20 líneas:")
        print("\n".join(eval_out.splitlines()[-20:]))
except subprocess.TimeoutExpired:
    print(f"  ! Eval timeout 600s — checkpoint probablemente inválido.")
print(f"  [{t_ts()}] Eval terminó en {time.time()-t_eval:.1f}s")


## 11. Export TF SavedModel

Usa `export_tflite_graph_tf2.py` para producir un `saved_model/` listo para TFLite Converter.

In [ ]:
banner("11. Export SavedModel")

EXPORT_CMD = [
    sys.executable,
    str(TF_MODELS_DIR / "research" / "object_detection" / "export_tflite_graph_tf2.py"),
    f"--pipeline_config_path={PIPELINE_PATH}",
    f"--trained_checkpoint_dir={TRAIN_RUN_DIR}",
    f"--output_directory={EXPORT_DIR}",
]
print(f"  [{t_ts()}] Ejecutando export...")
subprocess.run(EXPORT_CMD, check=True)

sm_dir = EXPORT_DIR / "saved_model"
assert sm_dir.exists(), f"saved_model no se creó en {sm_dir}"
print(f"  [{t_ts()}] SavedModel OK -> {sm_dir}")


## 12. Convertir a TFLite INT8 (PTQ + QAT graph)

Converter con representative dataset (400 muestras del val shuffleadas seed=42). I/O en `uint8` — listos para XNNPACK en Nano.

In [ ]:
banner("12. TFLite INT8")
import tensorflow as tf
import numpy as np
from tqdm.auto import tqdm

# Usar DATASET_ROOT (no DATASET_DIR) — Roboflow puede crear subcarpeta versionada
val_records = sorted(glob.glob(str(DATASET_ROOT / "valid" / "*.tfrecord")))
assert val_records, f"No hay tfrecords en {DATASET_ROOT}/valid."

def representative_dataset():
    raw = tf.data.TFRecordDataset(val_records).shuffle(2000, seed=EVAL_SEED).take(REPR_DATASET_SIZE)
    feature_desc = {"image/encoded": tf.io.FixedLenFeature([], tf.string)}
    for i, example in enumerate(tqdm(raw, total=REPR_DATASET_SIZE, desc="repr")):
        parsed = tf.io.parse_single_example(example, feature_desc)
        img = tf.io.decode_jpeg(parsed["image/encoded"], channels=3)
        img = tf.image.resize(img, (320, 320))
        img = tf.cast(img, tf.uint8)
        yield [tf.expand_dims(img, 0).numpy()]

print(f"  [{t_ts()}] Creando converter...")
converter = tf.lite.TFLiteConverter.from_saved_model(str(sm_dir))
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS,  # fallback para ops no-INT8
]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

print(f"  [{t_ts()}] Cuantizando con {REPR_DATASET_SIZE} muestras representativas...")
t_q = time.time()
tflite_int8 = converter.convert()
print(f"  [{t_ts()}] Cuantización terminó en {time.time()-t_q:.1f}s")

TFLITE_PATH = WORK_DIR / "detect_int8.tflite"
TFLITE_PATH.write_bytes(tflite_int8)
print(f"  [{t_ts()}] {TFLITE_PATH.name}: {len(tflite_int8)/1024:.1f} KB")


## 13. Smoke test del modelo TFLite

Carga el `.tflite` con `tf.lite.Interpreter`, corre inferencia sobre 1 tfrecord del test y reporta latencia + top-1 detección.

In [ ]:
banner("13. Smoke test")

interp = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
out = interp.get_output_details()

print(f"  Input  : shape={inp['shape']} dtype={inp['dtype']}")
for o in out: print(f"  Output : {o['name']:32s} shape={o['shape']} dtype={o['dtype']}")

# Usar DATASET_ROOT y fallback a valid si no hay test split
test_records = sorted(glob.glob(str(DATASET_ROOT / "test" / "*.tfrecord")))
if not test_records:
    test_records = sorted(glob.glob(str(DATASET_ROOT / "valid" / "*.tfrecord")))
    print(f"  (Sin test split, usando valid como fallback)")
assert test_records, "No hay tfrecords ni en test/ ni en valid/"

sample = next(iter(tf.data.TFRecordDataset(test_records).take(1)))
feature_desc = {"image/encoded": tf.io.FixedLenFeature([], tf.string)}
parsed = tf.io.parse_single_example(sample, feature_desc)
img = tf.io.decode_jpeg(parsed["image/encoded"], channels=3)
img = tf.image.resize(img, (320, 320))
img = tf.cast(img, tf.uint8).numpy()

# Warm-up + 5 inferences
interp.set_tensor(inp["index"], np.expand_dims(img, 0))
interp.invoke()
lats = []
for _ in range(5):
    interp.set_tensor(inp["index"], np.expand_dims(img, 0))
    t0 = time.perf_counter()
    interp.invoke()
    lats.append((time.perf_counter() - t0) * 1000)

print(f"\n  Latencia x86 (NO es proxy de Jetson — Maxwell ARM es distinto):")
print(f"    min={min(lats):.1f}ms p50={sorted(lats)[2]:.1f}ms max={max(lats):.1f}ms")
print(f"  [{t_ts()}] Smoke test OK")


## 14. Manifest JSON + persistir a Drive

Genera un `manifest.json` con todo lo necesario para reproducir + reporta en informe IEEE.

In [ ]:
banner("14. Manifest + persistencia")
import hashlib

md5 = hashlib.md5(TFLITE_PATH.read_bytes()).hexdigest()
sha256 = hashlib.sha256(TFLITE_PATH.read_bytes()).hexdigest()

manifest = {
    "track": "A",
    "model": "ssd_mobilenet_v2_fpnlite_320x320 -> tflite_int8_ptq",
    "platform_train": PLATFORM,
    "stack_train": {
        "python": sys.version.split()[0],
        "tensorflow": __import__("tensorflow").__version__,
        "tf_models_official": "2.15.0",
        "via_condacolab": IS_COLAB,
    },
    "stack_runtime_target": {
        "device": "Jetson Nano B01",
        "jetpack": "4.6.1",
        "python": "3.6.9",
        "tensorflow": "2.5.0+nv21.8",
        "inference": "TFLite Interpreter + XNNPACK + NEON (CPU)",
    },
    "roboflow": {"workspace": WORKSPACE, "project": PROJECT, "version": VERSION,
                 "format": "tfrecord", "resize": "fit-black 320x320"},
    "training": {
        "num_train_steps": NUM_TRAIN_STEPS,
        "batch_size": BATCH_SIZE,
        "eval_seed": EVAL_SEED,
        "wallclock_minutes": (time.time() - state["started_at"]) / 60,
    },
    "quantization": {
        "scheme": "PTQ post-train con representative_dataset",
        "reason": "graph_rewriter QAT no funciona en TF2 (issue tensorflow/models #9835)",
        "repr_dataset_size": REPR_DATASET_SIZE,
        "input_dtype": "uint8",
        "output_dtype": "uint8",
    },
    "artifact": {
        "filename": TFLITE_PATH.name,
        "size_kb": round(len(TFLITE_PATH.read_bytes()) / 1024, 1),
        "md5": md5,
        "sha256": sha256,
    },
    "metrics": {"map50_fp32_val": map50},
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
}

MANIFEST_PATH = WORK_DIR / "manifest_track_a.json"
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

# Copy a Drive
print(f"\n  [{t_ts()}] Persistiendo a {PERSIST_ROOT}...")
shutil.copy(TFLITE_PATH, PERSIST_ROOT / TFLITE_PATH.name)
shutil.copy(MANIFEST_PATH, PERSIST_ROOT / MANIFEST_PATH.name)
shutil.copy(PIPELINE_PATH, PERSIST_ROOT / PIPELINE_PATH.name)
# Copia incremental de los checkpoints de RUNS_DIR -> Drive (para resumes futuros)
runs_persist = PERSIST_ROOT / "runs"
shutil.copytree(TRAIN_RUN_DIR, runs_persist / TRAIN_RUN_DIR.name, dirs_exist_ok=True)
print(f"  [{t_ts()}] OK. Lista de archivos en {PERSIST_ROOT}:")
for p in sorted(PERSIST_ROOT.glob("*")):
    print(f"    {p.name:32s}{p.stat().st_size/1024:.1f} KB")

print(f"\n  ✓ detect_int8.tflite listo para scp al Jetson Nano:")
print(f"    {PERSIST_ROOT / TFLITE_PATH.name}")
